# IndoFraud-Synth & Skew-Aware FL — Google Colab Runner

This notebook runs the multi-seed federated ensemble experiments on **Google Colab** using:
- **`uv`** for fast (~15s) dependency installation while **preserving Colab's preinstalled CUDA PyTorch**
- **GPU acceleration** (`--device auto`) for PyTorch MLP and XGBoost (`tree_method=hist, device=cuda`), plus all-core CPU threading for CatBoost
- **Automatic Parquet caching** so 6M-row datasets load in ~2s instead of ~35s
- **Cross-condition FL model caching** (`_model_cache/`) so identical XGBoost/CatBoost/MLP trainings are never repeated across ablation conditions
- **Disconnect-safe resumption** via Google Drive + `--skip-existing`

### Recommended Runtime
- **Colab Pro:** `L4` or `A100` GPU + **High-RAM** (runs full uncapped `configs/default.yaml` comfortably)
- **Colab Free:** `T4` GPU (12.7 GB RAM) — set `TRAIN_CAP = 150000` below to stay safely within RAM limits (or use `notebooks/kaggle_runner.ipynb` for free 30 GB RAM + T4×2).

In [ ]:
# 1. Inspect hardware & install uv
import os, shutil, subprocess

!nvidia-smi || true
!echo "CPUs: $(nproc)" && free -h

if shutil.which("uv") is None:
    !curl -LsSf https://astral.sh/uv/install.sh | sh
    os.environ["PATH"] = os.path.expanduser("~/.local/bin") + ":" + os.environ["PATH"]

!uv --version

In [ ]:
# 2. Mount Google Drive & sync repository to fast local SSD (/content)
from google.colab import drive
drive.mount("/content/drive")

import os, pathlib

# --- EDIT THESE PATHS TO MATCH YOUR SETUP ---
# Option A: Clone from GitHub (leave empty if syncing code from Google Drive)
REPO_URL = ""  # e.g., "https://github.com/<your-user>/IndoFraudSynth.git"
# Option B: Folder in Google Drive containing your repo (used if REPO_URL is empty)
DRIVE_REPO_DIR = "/content/drive/MyDrive/IndoFraudSynth"

# Persistent output & data directories on Google Drive (survives Colab disconnects)
DRIVE_ROOT = "/content/drive/MyDrive/IndoFraudSynth_runs"
DRIVE_DATA = f"{DRIVE_ROOT}/data"
DRIVE_RESULTS = f"{DRIVE_ROOT}/results"
WORK_DIR = "/content/IndoFraudSynth"

os.makedirs(DRIVE_DATA, exist_ok=True)
os.makedirs(DRIVE_RESULTS, exist_ok=True)

if REPO_URL:
    if not os.path.exists(WORK_DIR):
        !git clone {REPO_URL} {WORK_DIR}
    else:
        !git -C {WORK_DIR} pull --ff-only
else:
    assert os.path.isdir(DRIVE_REPO_DIR), (
        f"Upload your IndoFraudSynth folder to {DRIVE_REPO_DIR} or set REPO_URL above."
    )
    !rsync -a --exclude='.venv' --exclude='.git' --exclude='results' --exclude='data' "{DRIVE_REPO_DIR}/" "{WORK_DIR}/"

%cd {WORK_DIR}
!ls -la

In [ ]:
# 3. Install only missing packages with uv (preserves Colab's CUDA PyTorch & XGBoost)
import importlib.util, os, subprocess

# Only install what's missing; never downgrade Colab's preinstalled CUDA torch/xgboost/numpy
need = []
for mod, pkg in [
    ("catboost", "catboost>=1.2.7"),
    ("flwr", "flwr>=1.13.0"),
    ("xgboost", "xgboost>=2.1.1"),
    ("pyarrow", "pyarrow>=15.0"),
    ("yaml", "pyyaml"),
    ("pytest", "pytest"),
]:
    if importlib.util.find_spec(mod) is None:
        need.append(pkg)

if need:
    print("Installing missing packages with uv:", need)
    subprocess.run(
        ["uv", "pip", "install", "--system", "--only-binary=:all:", *need],
        check=True,
    )
else:
    print("All required packages already installed.")

# Verify GPU + libraries
import torch, xgboost as xgb, catboost, numpy as np
print(f"PyTorch {torch.__version__} | CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print(f"XGBoost {xgb.__version__} | CatBoost {catboost.__version__} | CPUs: {os.cpu_count()}")

# Quick XGBoost CUDA sanity check
if torch.cuda.is_available():
    dm = xgb.DMatrix(np.random.randn(256, 8).astype(np.float32), label=np.random.randint(0, 2, 256))
    bst = xgb.train({"tree_method": "hist", "device": "cuda", "objective": "binary:logistic"}, dm, num_boost_round=2)
    print("XGBoost CUDA test: OK")

In [ ]:
# 4. Prepare dataset (auto-reuses existing data + Parquet cache on Google Drive)
import os

# Set to "full" (configs/default.yaml, ~6M rows) or "quick" (configs/quick.yaml, ~240k rows)
SYNTH_SCALE = "full"
DATA_DIR = f"{DRIVE_DATA}/{SYNTH_SCALE}"
CONFIG_PATH = "configs/default.yaml" if SYNTH_SCALE == "full" else "configs/quick.yaml"

if not os.path.exists(f"{DATA_DIR}/transactions.csv.gz"):
    print(f"Generating IndoFraud-Synth ({SYNTH_SCALE}) -> {DATA_DIR} ...")
    !python -m indofraud_synth --config {CONFIG_PATH} --out {DATA_DIR}
else:
    print(f"Found existing dataset at {DATA_DIR}")

# Pre-warm Parquet cache once so every seed/condition loads in ~2s
from fl_experiments.data import load_indofraud
import time
t0 = time.perf_counter()
_df = load_indofraud(DATA_DIR)
print(f"Loaded {len(_df):,} rows in {time.perf_counter() - t0:.1f}s (Parquet cached: {os.path.exists(f'{DATA_DIR}/transactions.parquet')})")
del _df

In [ ]:
# 5. Run Multi-Seed Experiment Suite (disconnect-safe via --skip-existing + model cache)
#
# TIP: To run even faster across multiple Colab sessions in parallel, shard SEEDS:
#   Session A: SEEDS = "1,2,3"
#   Session B: SEEDS = "4,5"
# Both write to the same OUT_DIR on Drive; run Cell 6 after both finish.

DATASET = "indofraud"   # indofraud | ulb | ieee_cis | sparkov
PARTITIONS = "profile"  # e.g. "profile,equal,dirichlet:0.1,dirichlet:0.5,dirichlet:5.0"
CONDITIONS = "baseline,normalized,normalized_fedprox,normalized_fedbn,normalized_size_fedbn"
SEEDS = "1,2,3,4,5"
TRAIN_CAP = 200000      # 0 = uncapped (High-RAM only); 150000-200000 safe on Free T4 (12.7 GB RAM)
PATIENCE = 3            # 3 saves ~35% rounds vs 5; use 5 for strict default.yaml match
MLP_BATCH_SIZE = 4096   # 4096 saturates T4/L4/A100 GPU well (use 1024 for exact default)
OUT_DIR = f"{DRIVE_RESULTS}/suite_{DATASET}"

!python -m fl_experiments.run_suite \
    --config {CONFIG_PATH} \
    --dataset {DATASET} \
    --data "{DATA_DIR}" \
    --engine local \
    --device auto \
    --train-cap {TRAIN_CAP} \
    --patience {PATIENCE} \
    --mlp-batch-size {MLP_BATCH_SIZE} \
    --partitions "{PARTITIONS}" \
    --conditions "{CONDITIONS}" \
    --seeds "{SEEDS}" \
    --out "{OUT_DIR}" \
    --skip-existing

In [ ]:
# 6. Re-compute Friedman-Nemenyi stats (useful after merging sharded sessions) & preview results
import os
from IPython.display import display, Markdown, Image

SUITE_PART_DIR = f"{OUT_DIR}/profile"
!python -m fl_experiments.stats --suite-dir "{SUITE_PART_DIR}" --metric pooled_ap

summary_md = os.path.join(SUITE_PART_DIR, "summary.md")
cd_png = os.path.join(SUITE_PART_DIR, "cd_diagram.png")
if os.path.exists(summary_md):
    display(Markdown(open(summary_md).read()))
if os.path.exists(cd_png):
    display(Image(filename=cd_png))